# Ocular adversarial XAI: the next run (data check, eye masks, verification, experiment v2, adaptive attacks)

One Kaggle session for everything the 31 Oct re-check asked for, with both GPUs and the CPU busy at the same time.

**Before you start**
1. The six PRs (runner-v2, analysis-v2, adaptive attacks, eye masks, verification, data check) must be merged into `BRANCH` (cell 1, default `main`). Cell 6 stops if a script, module or flag is missing.
2. Settings: accelerator **GPU T4 x2**, Internet **on** (the eye-mask weights come from Hugging Face), secret `github_token` attached.
3. Inputs: `mubashirulhassan00/visob-augmented-recent` and `nandininautiyal/ocular-xception-arcface-full-baseline`.
4. Keep this notebook **private**. Never commit or upload `results.zip`: it holds results derived from VISOB.

**Workflow**
- First run all cells interactively with `RUN_FULL = False` (about 1 h 15 min): setup, checks, the manifest, the clean evaluation, the eye masks, a smoke run of every script, time estimates for the full runs, and a checklist of the smoke outputs.
- Then set `RUN_FULL = True` (cell 2) and run with **Save Version > Save & Run All**, so the outputs are kept (about 4-5 h). The checks and smoke runs are repeated, and the notebook stops before the full runs if a check fails or the estimate does not fit in Kaggle's 12-hour session. You can go straight to this step if you trust the setup.
- Download `results.zip` from the Output panel. Never import TensorFlow in a notebook cell: every TensorFlow script runs in its own process.

**Plan** (times from the 25 and 27 Oct runs; the smoke runs give this session's estimates)

| Cell | GPU 0 | GPU 1 | CPU |
|---|---|---|---|
| 7. Data (~17 min) | | | 01 manifest, 02 random split and open-set list |
| 8. Phase A (~40 min) | 04 clean evaluation (~31 min) | 17 eye masks: smoke, then all 13,868 test images; 15 smoke | |
| 10. Phase B (~10 min) | 05 smoke (8 images) | 14 smoke (the same 8 images) | 16 smoke, then 12 on the 05 smoke rows |
| 12. Phase C (2-3 h, `RUN_FULL`) | 05 full: 600 images, eye-mask regions, maps saved | 14 full (300 images), then 15 P1 + P3 | 16 data check (low priority) |
| 13. Phase D (~10 min, `RUN_FULL`) | | | 12 analysis of the new rows |

## 1-3. Setup, settings and the commands of the session

In [ ]:
# 1. Setup: clone the repo at BRANCH, install the pinned tf-keras, record the commit
import os, subprocess, time
SESSION_START = time.time()   # Kaggle ends a session after 12 h; the job runner (cell 4) watches this clock

BRANCH = 'main'   # the six PRs (runner-v2, analysis-v2, adaptive, eye-masks, verification, data-check) must be in it
CKPT = '/kaggle/input/datasets/nandininautiyal/ocular-xception-arcface-full-baseline/trained_models/Xception_ArcFace_full.keras'

!pip install -q --no-deps tf-keras==2.20.1

from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret('github_token')
REPO = '/tmp/ocular-adv-xai'
url = f'https://{tok}@github.com/nandininautiyal/ocular-adv-xai.git'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/nandininautiyal/ocular-adv-xai.git
else:
    # a reused session may have another branch checked out: always land exactly on BRANCH
    !git -C {REPO} fetch -q {url} {BRANCH}
    !git -C {REPO} checkout -q -f -B {BRANCH} FETCH_HEAD
%cd {REPO}
!git log -3 --oneline
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('branch:', BRANCH, '| commit:', COMMIT, '| checkpoint found:', os.path.exists(CKPT))
!nvidia-smi -L

In [ ]:
# 2. Settings: the only cell to edit
RUN_FULL = False      # False: checks, Phase A, smoke runs and time estimates. True: also the full runs and the
                      # analysis (use Save Version > Save & Run All)
TAG = 'v2'            # tag of the full runs' outputs: rows_v2.csv, summary_adaptive_v2.json, ...
SMOKE = 'smoke'       # tag of the smoke runs
LEVELS = [1, 2, 4]    # L-inf budgets in 8-bit levels: 1/255, 2/255 and 4/255 (the defaults of 05 and 14)
EPSILONS = ','.join(f'{n / 255:.7f}' for n in LEVELS)  # the same budgets as numbers (0.0039216, ...), for 14
PER_CLASS_05 = 2      # images per class in 05: 600 images, the same 600 as the 25 and 27 Oct runs
ATTACKS_05 = 'fgsm,bim,pgd,mim,cw,patch,monocle'
CW_KAPPA = 5.0        # CW confidence margin in logits (05's default); 0 gives the minimal attack of the 27 Oct run
JPEG_QUALITY, BITS = 75, 4   # the defences of 05 and 14 (their defaults)
MASK_SOURCE = 'eye_masks'    # 'eye_masks': regions from 17's learned masks (zero-shot, not yet checked against hand
                             # labels; also moves the monocle onto the mask's iris); 'heuristic': src/xai/masks.py
ADAPTIVE_STEPS = 20   # gradient steps of every attack in 14
BATCH = 16            # images per batch in 05 and 14; lower to 8 if a GPU runs out of memory
DATA_CHECK_WORKERS = 3     # processes for 16, which runs at low priority next to the GPU jobs (4 CPU cores)
WEIGHTS = None        # path of an attached combine.pth, to skip downloading the eye-mask weights
EXPORT_LABEL_PACK = True   # 18's pack of 60 images to hand-label (team agreed 1 Nov); keep label_pack.zip private
HARD_STOP_H = 11.25   # Kaggle stops a session after 12 h; any job still running at this point is stopped, so that
                      # the version can still save its outputs
ZIP_MAX_MB = 100      # files larger than this stay out of results.zip (download them from the Output panel)

W = '/kaggle/working'
LOGS = f'{W}/logs'
CORRECT = f'{W}/eval_clean/correct_test_full.csv'
EXP, ADAPTIVE, VERIF, CHECK = f'{W}/experiment', f'{W}/adaptive', f'{W}/verification', f'{W}/data_check'
MASKS = f'{W}/eye_masks/masks_test.npz'
OPEN_SET = f'{W}/splits/open_set_test.csv'

In [ ]:
# 3. Every command of the session, in one place (nothing runs in this cell)
ck = f"--ckpt '{CKPT}'"
weights = '' if WEIGHTS is None else f"--weights '{WEIGHTS}'"
regions = f'--concept-masks --mask-source {MASKS}' if MASK_SOURCE == 'eye_masks' else '--concept-masks'
levels = ','.join(str(n) for n in LEVELS)
exp05 = (f'{ck} --correct-csv {CORRECT} --attacks {ATTACKS_05} --levels {levels} --cw-kappa {CW_KAPPA} '
         f'--jpeg-quality {JPEG_QUALITY} --bits {BITS} --batch-size {BATCH} {regions} --save-maps')
adaptive14 = (f'{ck} --correct-csv {CORRECT} --epsilons {EPSILONS} --jpeg-quality {JPEG_QUALITY} --bits {BITS} '
              f'--steps {ADAPTIVE_STEPS} --batch-size {BATCH}')
verif15 = f'{ck} --modes p1,p3 --manifest {W}/manifest_raw.csv --p3-list {OPEN_SET}'
check16 = f'nice -n 10 python -u scripts/16_check_dataset.py --images-csv {CORRECT} --workers {DATA_CHECK_WORKERS}'
CMD = {
    # data, in the foreground (cell 7)
    '01_manifest': f'python -u scripts/01_build_manifest.py --out {W}/manifest_raw.csv',
    '02_splits':   'python -u scripts/02_make_splits.py',      # random protocol, test split pinned
    '02_open_set': f'python -u scripts/02_make_splits.py --protocol open_set --write-split-csvs {W}/splits',
    # Phase A (cell 8)
    '04_eval':     f'python -u scripts/04_eval_clean.py --split-dir /tmp/ocular_recognition --n-classes 300 {ck} '
                   f'--manifest {W}/manifest_split.csv --tag full',
    '17_smoke':    f'python -u scripts/17_make_eye_masks.py --all-test --max-images 40 {weights} --tag {SMOKE}',
    '17_masks':    f'python -u scripts/17_make_eye_masks.py --all-test {weights} --tag test',
    '15_smoke':    f'python -u scripts/15_eval_verification.py {verif15} --max-subjects 10 --n-boot 100 --tag {SMOKE}',
    # Phase B (cell 10): 05 and 14 use the same 8 images (seed 1); the full runs leave them out
    '05_smoke':    f'python -u scripts/05_run_experiment.py {exp05} --max-images 8 --seed 1 --tag {SMOKE}',
    '14_smoke':    f'python -u scripts/14_adaptive_attacks.py {adaptive14} --per-class 2 --max-images 8 --seed 1 '
                   f'--tag {SMOKE}',
    '16_smoke':    f'{check16} --max-buckets 20 --max-images 20 --out-dir {W}/data_check_smoke',
    '12_smoke':    f'python -u scripts/12_analyse.py --experiment-rows {EXP}/rows_{SMOKE}.csv --n-boot 100 '
                   f'--n-perm 99 --out-dir {W}/analysis_{SMOKE}',
    # Phase C (cell 12): seed 2 as on 25 and 27 Oct; 14 takes the first image per class of 05's sample
    '05_full':     f'python -u scripts/05_run_experiment.py {exp05} --per-class {PER_CLASS_05} --seed 2 '
                   f'--exclude-from {EXP}/images_{SMOKE}.csv --tag {TAG}',
    '14_full':     f'python -u scripts/14_adaptive_attacks.py {adaptive14} --per-class 1 --seed 2 '
                   f'--exclude-from {EXP}/images_{SMOKE}.csv --tag {TAG}',
    '15_full':     f'python -u scripts/15_eval_verification.py {verif15} --save-embeddings --tag {TAG}',
    '16_full':     f'{check16} --out-dir {CHECK}',
    # Phase D (cell 13)
    '12_full':     f'python -u scripts/12_analyse.py --experiment-rows {EXP}/rows_{TAG}.csv --out-dir {W}/analysis_{TAG}',
    '18_export':   f'python -u scripts/18_validate_masks.py --export-sample --csv {CORRECT} --masks {MASKS} '
                   f'--out-dir {W}/label_pack --zip',
}
# a background step runs only if these steps succeeded (in its own phase or an earlier one)
NEEDS = {'17_masks': ['17_smoke'],
         '05_smoke': ['04_eval'] + (['17_masks'] if MASK_SOURCE == 'eye_masks' else []),
         '14_smoke': ['04_eval'], '16_smoke': ['04_eval'],
         '05_full': ['05_smoke'], '14_full': ['05_smoke', '14_smoke'], '15_full': ['15_smoke'], '16_full': ['16_smoke']}
for name, cmd in CMD.items():
    print(f'{name:12} {cmd}')

## 4-5. Helpers and checks

The job runner starts one chain of steps per GPU (and one on the CPU) in the background. Each step writes `logs/<step>.log` and `logs/<step>.exit`, runs only if the steps it needs succeeded, and is stopped at `HARD_STOP_H` so the version can still save its outputs. The check functions are used after the smoke runs (cell 11) and at the end (cell 15).

In [ ]:
# 4. Helpers: foreground commands, background jobs (one GPU each), the 12-hour guard
import glob, hashlib, json, shutil, signal
import numpy as np
import pandas as pd
os.makedirs(LOGS, exist_ok=True)
FG = {}      # exit status of the foreground steps run with sh(..., check=False)

def hours():
    return (time.time() - SESSION_START) / 3600

def sh(cmd, check=True):
    # run cmd from the repo with its output shown here; check: stop the notebook if it fails
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, executable='/bin/bash', text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    code = p.wait()
    assert code == 0 or not check, f'exit code {code}: {cmd}'
    return code

def py(snippet):
    # (exit code, last output line) of a Python snippet run in a fresh process (keeps TensorFlow out of the notebook)
    r = subprocess.run(['python', '-c', snippet], cwd=REPO, capture_output=True, text=True)
    out = r.stdout.strip().splitlines() or r.stderr.strip().splitlines() or ['']
    return r.returncode, out[-1]

def load(path):
    with open(path) as f:
        return json.load(f)

def tail(path, n=12):
    try:
        with open(path, errors='ignore') as f:
            lines = [line.rstrip() for line in f if line.strip()]
    except FileNotFoundError:
        return ['(no log)']
    return lines[-n:] or ['(empty log)']

def status(name):
    # 0 ok, another number failed, 'skipped' (a step it needs failed), None (not run, or stopped)
    try:
        text = open(f'{LOGS}/{name}.exit').read().strip()
    except FileNotFoundError:
        return None
    return int(text) if text.lstrip('-').isdigit() else text

def launch(label, names, gpu=None):
    # run the steps CMD[name] one after the other in a background job on one GPU (gpu=None: CPU only)
    lines = [f'cd {REPO}', f"export CUDA_VISIBLE_DEVICES={'' if gpu is None else gpu}",
             'export PYTHONUNBUFFERED=1 TF_CPP_MIN_LOG_LEVEL=1']
    for name in names:
        for ext in ('log', 'exit'):
            if os.path.exists(f'{LOGS}/{name}.{ext}'):
                os.remove(f'{LOGS}/{name}.{ext}')
        needs = NEEDS.get(name, [])
        ok = ' && '.join(f'[ "$(cat {LOGS}/{n}.exit 2>/dev/null)" = 0 ]' for n in needs) or 'true'
        lines.append(f'if {ok}; then {CMD[name]} > {LOGS}/{name}.log 2>&1; echo $? > {LOGS}/{name}.exit; '
                     f'else echo "skipped: needs {" ".join(needs)}" > {LOGS}/{name}.log; '
                     f'echo skipped > {LOGS}/{name}.exit; fi')
    proc = subprocess.Popen(['bash', '-c', '\n'.join(lines)], start_new_session=True)
    print(f'{label}: {" -> ".join(names)}', flush=True)
    return {'label': label, 'names': names, 'proc': proc}

def stop(jobs):
    # stop the running jobs with all their processes (each job is its own process group): SIGTERM, then SIGKILL
    running = [j for j in jobs if j['proc'].poll() is None]
    for sig in (signal.SIGTERM, signal.SIGKILL):
        for job in running:
            try:
                os.killpg(job['proc'].pid, sig)
            except ProcessLookupError:     # every process of that job has ended
                pass
        if sig == signal.SIGTERM:
            time.sleep(30)

def wait(jobs, every=60, also=()):
    # wait for jobs, printing the last log line of each running step every few seconds (also for the jobs in
    # also, which keep running); stop every job at HARD_STOP_H; print and return {step: status}
    while any(j['proc'].poll() is None for j in jobs):
        time.sleep(every)
        if hours() > HARD_STOP_H:
            print(f'{hours():.2f} h: past HARD_STOP_H = {HARD_STOP_H}, stopping every job so the outputs can be saved')
            stop(list(jobs) + list(also))
            break
        parts = []
        for j in list(jobs) + list(also):
            step = next((n for n in j['names'] if status(n) is None), None) if j['proc'].poll() is None else None
            parts.append(f"{j['label']} {step}: {tail(f'{LOGS}/{step}.log', 1)[0][:150]}" if step
                         else f"{j['label']}: done")
        print(time.strftime('%H:%M'), f'({hours():.1f} h)', ' | '.join(parts), flush=True)
    codes = {n: status(n) for j in jobs for n in j['names']}
    for name, code in codes.items():
        word = {0: 'ok', None: 'NOT RUN (or stopped)', 'skipped': 'SKIPPED (a step it needs failed)'}.get(
            code, f'FAILED (exit {code})')
        print(f'\n---- {name}: {word}')
        print('\n'.join(tail(f'{LOGS}/{name}.log', 12 if code == 0 else 40)))
    return codes

In [ ]:
# 5. Checks of each script's outputs; each returns [(check, passed, detail, hard)], hard = must pass
N_CORRECT = 13585   # correct test images of the checkpoint (9 Oct evaluation, repeated on 25 Oct)
CORRECT_FINGERPRINT = '1998d6fb6051a4e52bf742884e1e296c2b3d226ded2220a326c3b3c346506503'  # their link paths, in order
SAMPLE_FINGERPRINT = '5f6a2d91039bdff39b7caf1e9154bf6ca867e822a081c6959eeeb71987305cdd'   # the 600 images of 25/27 Oct
CLASS_NAMES = ['skin', 'iris', 'sclera', 'lid', 'caruncle', 'brow']   # eye-mask file (contract 1)
HEURISTIC = ['pupil', 'iris', 'sclera', 'periocular']
PERIOCULAR_AREA = 1 - 20155 / 65536   # outside the fixed ellipse of src/xai/masks.py, the same on every image

def fingerprint(lines):
    return hashlib.sha256('\n'.join(lines).encode()).hexdigest()

def same_commit(value):
    # the commit a script recorded is this session's (some scripts store a short hash; a dirty mark fails)
    return isinstance(value, str) and len(value) >= 7 and COMMIT.startswith(value)

def budget_report(rows):
    # L-inf of the scored images in 8-bit levels: noise and FGSM/BIM/PGD/MIM must sit on whole levels, never above
    # the budget; every attacked image (CW, patch and monocle too) must be an 8-bit image
    budgeted = rows[(rows.condition == 'noise')
                    | ((rows.condition == 'attack') & rows.attack.isin(['fgsm', 'bim', 'pgd', 'mim']))]
    attacked = rows[rows.condition == 'attack']
    lev, budget = budgeted.pert_linf * 255, (budgeted.epsilon * 255).round()
    whole = lambda x: bool((np.abs(x - x.round()) < 1e-3).all())
    return {'on_grid': whole(lev) and whole(attacked.pert_linf * 255), 'within': bool((lev <= budget + 1e-3).all()),
            'at_budget': float((np.abs(lev - budget) < 1e-3).mean()), 'n': len(budgeted)}

def checks_04():
    s = load(f'{W}/eval_clean/summary_full.json')
    correct = pd.read_csv(CORRECT)
    same = fingerprint(correct.link_path) == CORRECT_FINGERPRINT
    return [('04: 13,585 of 13,868 test images correct (within 5)',
             s['n_images'] == 13868 and abs(s['n_correct'] - N_CORRECT) <= 5, f"{s['n_correct']} correct", True),
            ('04: the same correct list as on 9 and 25 Oct', same,
             '05 can draw the 25/27 Oct images' if same else '05 will draw other images: unpaired with 27 Oct', False)]

def checks_masks(correct):
    hard = MASK_SOURCE == 'eye_masks'
    if not os.path.exists(MASKS):
        return [('17: eye-mask file', False, f'{MASKS} missing', hard)]
    with np.load(MASKS, allow_pickle=False) as z:    # labels (909 MB) are not loaded here; 05 checks them
        rel = {str(r) for r in z['relpaths']}
        names = [str(c) for c in z['class_names']]
        meta = json.loads(str(z['meta'][()]))
    s = load(f'{W}/eye_masks/mask_summary_test.json')
    absent = set(correct.relpath.astype(str).str.replace('\\', '/', regex=False)) - rel
    return [('17: masks for all 13,868 test images, contract class names', len(rel) == 13868 and names == CLASS_NAMES,
             f'{len(rel)} images, {names}', hard),
            ('17: a mask for every correct image', not absent, f'{len(absent)} missing', hard),
            ('17: commit and clean checkout', same_commit(meta.get('code_commit')) and meta.get('code_dirty') is False,
             str(meta.get('code_commit')), hard),
            ('17: empty-iris rate below 2 %', s['empty_iris_rate'] < 0.02, f"{s['empty_iris_rate']:.2%}", False)]

def checks_05(tag, n_images):
    files = [f'{EXP}/{stem}_{tag}.{ext}' for stem, ext in
             (('rows', 'csv'), ('images', 'csv'), ('summary', 'json'), ('summary', 'csv'), ('maps', 'npz'))]
    missing = [f for f in files if not os.path.exists(f)]
    out = [(f'05 {tag}: rows, images, summary and maps files', not missing, ', '.join(missing), True)]
    if missing:
        return out
    s, images = load(f'{EXP}/summary_{tag}.json'), pd.read_csv(f'{EXP}/images_{tag}.csv')
    rows = pd.read_csv(f'{EXP}/rows_{tag}.csv', low_memory=False)
    out.append((f'05 {tag}: {n_images} images', len(images) == s['n_images'] == n_images,
                f"{len(images)} images of {images.true_idx.nunique()} classes", True))
    out.append((f'05 {tag}: commit and clean checkout', same_commit(s.get('git_commit')) and s.get('git_dirty') is False,
                f"{s.get('git_commit')}, dirty {s.get('git_dirty')}", True))
    out.append((f'05 {tag}: budgets of {LEVELS} levels, 8-bit images', s.get('levels') == LEVELS
                and s.get('quantised_8bit') is True, f"levels {s.get('levels')}", True))
    b = budget_report(rows)
    out.append((f'05 {tag}: budgets exact (whole 8-bit levels, never above the budget)', b['on_grid'] and b['within'],
                f"{b['n']} noise/FGSM/BIM/PGD/MIM rows, {b['at_budget']:.1%} exactly at the budget", True))
    cw, success = s.get('cw') or {}, s.get('attack_success_rate', {})
    out.append((f'05 {tag}: CW with kappa {CW_KAPPA:g}, success judged after 8-bit rounding',
                cw.get('kappa') == CW_KAPPA and cw.get('success_after_8bit_rounding') is True, str(cw), True))
    want = {'clean', 'noise', 'defence_on_clean', 'noise+defence', 'attack', 'attack+defence'}
    out.append((f'05 {tag}: all six conditions, incl. noise+defence', want <= set(rows.condition)
                and bool(s.get('noise_defence_flip_rate')), ', '.join(sorted(set(rows.condition))), True))
    with np.load(f'{EXP}/maps_{tag}.npz') as z:
        n_maps = len(z.files)
    ids = rows.row_id.is_unique and rows.row_id.min() == 0 and rows.row_id.max() == len(rows) - 1
    out.append((f'05 {tag}: row ids 0..n-1 and one map per row', bool(ids) and n_maps == len(rows),
                f'{len(rows)} rows, {n_maps} maps', True))
    regions = s.get('regions') or []
    expect = (MASKS, CLASS_NAMES) if MASK_SOURCE == 'eye_masks' else ('heuristic', HEURISTIC)
    out.append((f'05 {tag}: regions from {MASK_SOURCE}', (s.get('mask_source'), regions) == expect,
                f"{s.get('mask_source')}: {regions}", True))
    clean = rows[rows.condition == 'clean']
    mass = clean[[f'mass_{r}' for r in regions if f'mass_{r}' in rows]].dropna()
    sums = mass.sum(axis=1)
    # unlabelled (255) mask pixels keep their share of the map, so masses then sum to less than 1
    areas = [f'area_{r}' for r in regions]
    full = clean.loc[mass.index, areas].sum(axis=1).ge(1 - 1e-4) if set(areas) <= set(rows) else True
    ok = np.where(full, np.isclose(sums, 1, atol=1e-4), (sums > 0) & (sums <= 1 + 1e-4))
    out.append((f'05 {tag}: clean-map region masses sum to 1 (less with unlabelled pixels)',
                len(mass) > 0 and bool(np.all(ok)), f'{len(mass)} clean rows, sums {sums.min():.4f} to {sums.max():.4f}',
                True))
    if MASK_SOURCE == 'eye_masks':
        with np.load(MASKS, allow_pickle=False) as z:
            absent = set(images.relpath) - {str(r) for r in z['relpaths']}
        out.append((f'05 {tag}: no missing eye masks', not absent, f'{len(absent)} images without a mask', True))
    placements = s.get('monocle_placement_counts') or {}
    out.append((f'05 {tag}: a monocle placement for every image', sum(placements.values()) == n_images,
                str(placements), True))
    rate = success.get('cw', {}).get('rate', float('nan'))
    out.append((f'05 {tag}: CW success after 8-bit rounding at least 0.9 (27 Oct: 0.625)', rate >= 0.9, f'{rate}', False))
    if tag == TAG and PER_CLASS_05 == 2:
        same = fingerprint([f'{i},{p}' for i, p in zip(images.image_id, images.link_path)]) == SAMPLE_FINGERPRINT
        out.append((f'05 {tag}: the same 600 images as the 25 and 27 Oct runs (paired comparison)', same, '', False))
    return out

def checks_14(tag, images_05):
    path, spath = f'{ADAPTIVE}/rows_adaptive_{tag}.csv', f'{ADAPTIVE}/summary_adaptive_{tag}.json'
    if not (os.path.exists(path) and os.path.exists(spath)):
        return [(f'14 {tag}: rows and summary files', False, f'{path} or {spath} missing', True)]
    rows, s = pd.read_csv(path), load(spath)
    n = len(s['attacks']) * len(s['epsilons']) * len(s['defences']) * s['n_images']
    lev = rows.pert_linf_levels
    out = [(f'14 {tag}: rows = images x attacks x budgets x defences', len(rows) == n,
            f"{len(rows)} rows, {s['n_images']} images", True),
           (f'14 {tag}: budgets of {LEVELS} levels, 8-bit images', s.get('budget_levels') == LEVELS
            and s.get('quantised_8bit') is True, str(s.get('budget_levels')), True),
           (f'14 {tag}: budgets exact (whole levels, never above the budget)',
            bool((np.abs(lev - lev.round()) < 1e-3).all() and (lev <= rows.budget_levels + 1e-3).all()),
            f'largest {lev.max():g} levels', True),
           (f'14 {tag}: commit and clean checkout', same_commit(s.get('git_commit')), str(s.get('git_commit')), True)]
    if images_05 is not None:
        out.append((f"14 {tag}: its images are among 05's (paired)", set(rows.link_path) <= set(images_05.link_path),
                    f'{rows.link_path.nunique()} images', True))
    after = s.get('success_after_defence', {})
    weaker = [k for k in after if k.startswith('bpda_pgd@') and k.endswith('+jpeg')
              and after[k]['rate'] < after.get(k.replace('bpda_pgd@', 'pgd@'), {}).get('rate', 0)]
    out.append((f'14 {tag}: BPDA at least as strong as non-adaptive PGD through JPEG', not weaker,
                ', '.join(weaker) or 'every budget', False))
    return out

def checks_15(tag, full):
    mpath, spath = f'{VERIF}/verification_metrics_{tag}.csv', f'{VERIF}/verification_summary_{tag}.json'
    if not (os.path.exists(mpath) and os.path.exists(spath)):
        return [(f'15 {tag}: metrics and summary files', False, f'{mpath} or {spath} missing', True)]
    m, s = pd.read_csv(mpath), load(spath)
    overall = m[m.subset == 'all'].set_index('mode')
    in_ci = all(np.isnan(r.eer_ci_low) or r.eer_ci_low <= r.eer <= r.eer_ci_high for r in overall.itertuples())
    p1 = s['per_mode'].get('p1', {})
    out = [(f'15 {tag}: P1 and P3 results', {'p1', 'p3'} <= set(overall.index), ', '.join(overall.index), True),
           (f'15 {tag}: EER between 0 and 0.5', bool(overall.eer.between(0, 0.5).all()),
            ', '.join(f'{k} EER {r.eer:.4f} [{r.eer_ci_low:.4f}, {r.eer_ci_high:.4f}]' for k, r in overall.iterrows()),
            True),
           (f'15 {tag}: EER inside its bootstrap CI', in_ci, '', False),
           (f'15 {tag}: commit and clean checkout', same_commit(s.get('git_commit')), str(s.get('git_commit')), True)]
    if full:
        out.append((f'15 {tag}: P1 = the 13,868 pinned test images of 300 subjects',
                    p1.get('n_images') == 13868 and p1.get('n_subjects') == 300,
                    f"{p1.get('n_images')} images, {p1.get('n_subjects')} subjects", True))
    return out

def checks_16(folder, n_area_images):
    if not os.path.exists(f'{folder}/summary.json'):
        return [(f'16 {os.path.basename(folder)}: summary.json', False, 'missing', True)]
    s = load(f'{folder}/summary.json')
    areas = s.get('areas', {})
    peri = areas.get('mean_area', {}).get('periocular', float('nan'))
    table = f"{folder}/{areas.get('output', 'missing.csv')}"
    cols = ['relpath', 'link_path', 'mask_source'] + [f'area_{r}' for r in HEURISTIC]
    has_cols = os.path.exists(table) and set(cols) <= set(pd.read_csv(table, nrows=1).columns)
    name = os.path.basename(folder)
    return [(f'16 {name}: commit and clean checkout', same_commit(s.get('git_commit')) and s.get('git_dirty') is False,
             str(s.get('git_commit')), True),
            (f'16 {name}: 134,258 images counted', s.get('counts', {}).get('images') == 134258,
             str(s.get('counts', {}).get('images')), True),
            (f'16 {name}: synthetic-frame verdict', bool(s.get('synthetic', {}).get('verdict')),
             str(s.get('synthetic', {}).get('verdict')), True),
            (f'16 {name}: test-train overlap table', 'overlap' in s, '', True),
            (f'16 {name}: per-image areas of {n_area_images} images (contract 3)',
             areas.get('images') == n_area_images and abs(peri - PERIOCULAR_AREA) < 1e-4 and has_cols,
             f"{areas.get('images')} images, mean periocular area {peri:.6f}", True)]

def checks_12(folder, hard):
    need = ['report_v2.md', 'region_shift_v2.csv', 'region_null.csv', 'heatmap_shift_by_outcome.csv',
            'restoration_by_outcome.csv']
    missing = [f for f in need if not os.path.exists(f'{folder}/{f}')]
    return [(f'12: v2 analysis in {os.path.basename(folder)}', not missing, ', '.join(missing), hard)]

def run_checks(*specs):
    # specs: (check function, its arguments...); a crash counts as a failed hard check
    out = []
    for fn, *args in specs:
        try:
            out += fn(*args)
        except Exception as e:
            out.append((f'{fn.__name__} crashed', False, f'{type(e).__name__}: {e}', True))
    return out

def report(checks):
    # print PASS / WARN / FAIL per check; return the failed hard checks
    for name, ok, detail, hard in checks:
        print(f"{'PASS' if ok else ('FAIL' if hard else 'WARN')}  {name}" + (f'  [{detail}]' if detail else ''))
    return [c for c in checks if c[3] and not c[1]]

## 6-7. Pre-flight checks and data

Cell 6 checks everything that can be checked before the GPUs start. Cell 7 builds the manifest (about 15 min; it walks the 134,258 images) and the pinned random split, writes the open-set list for 15, and confirms that the test split equals the training-time test split (the file saved with the checkpoint) and `splits/visob_test.csv`.

In [ ]:
# 6. Pre-flight: checkpoint, GPUs, the merged code and its flags, packages, data, eye-mask weights, disk
problems = []
if not os.path.exists(CKPT):
    problems.append(f'checkpoint not found: {CKPT} (add the input nandininautiyal/ocular-xception-arcface-full-baseline)')
n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines())
print('GPUs visible:', n_gpu)
if n_gpu < 2:
    problems.append(f'{n_gpu} GPU(s) visible: set the accelerator to GPU T4 x2')

# the six PRs: every script, module and flag used here must be on BRANCH
scripts = sorted({t for cmd in CMD.values() for t in cmd.split() if t.startswith('scripts/')})
modules = ['src/attacks/budget.py', 'src/attacks/adaptive.py', 'src/eval/verification.py', 'src/xai/seg_masks.py',
           'src/analysis/region_null.py']
missing = [p for p in scripts + modules if not os.path.exists(f'{REPO}/{p}')]
if missing:
    problems.append(f'{BRANCH} lacks {missing}: merge the six PRs first (or set BRANCH to a branch that has them)')
for path in scripts:
    if os.path.exists(f'{REPO}/{path}'):
        text = open(f'{REPO}/{path}').read()
        used = {t for cmd in CMD.values() if path in cmd.split()
                for t in cmd.split()[cmd.split().index(path) + 1:] if t.startswith('--')}
        lost = sorted(f for f in used if f'"{f}"' not in text and f"'{f}'" not in text)
        if lost:
            problems.append(f'{path} has no {lost}: merge the six PRs first')

# packages, each checked in its own process
code, tfv = py("import os; os.environ['TF_USE_LEGACY_KERAS'] = '1'; import importlib.metadata as m, tensorflow as tf; "
               "print(tf.__version__, m.version('tf-keras'), len(tf.config.list_physical_devices('GPU')))")
print('tensorflow, tf-keras, GPUs seen by TensorFlow:', tfv)
parts = tfv.split()
if code or len(parts) != 3 or not parts[0].startswith('2.20.') or parts[1] != '2.20.1':
    problems.append(f'TensorFlow 2.20 with tf-keras 2.20.1 expected (the checkpoint was trained with them): {tfv}')
code, tv = py('import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())')
print('torch, torchvision, CUDA:', tv)
if code or not tv.endswith('True'):
    problems.append(f'17 needs PyTorch and torchvision with CUDA: {tv}')
code, cv = py("import os, cv2; print(cv2.__version__, os.path.exists(cv2.data.haarcascades + 'haarcascade_eye.xml'))")
print('OpenCV, Haar eye cascade present:', cv)
major = cv.split('.')[0]
if code or not major.isdigit() or int(major) >= 5 or not cv.endswith('True'):
    problems.append(f'OpenCV < 5 with the Haar eye cascade expected (Monocle Blending): {cv}')

# data, the training-time split saved with the checkpoint, the eye-mask weights, disk space
VISOB = next((os.path.join(r, 'VISOB_Augmented_Recent') for r, dirs, _ in os.walk('/kaggle/input')
              if 'VISOB_Augmented_Recent' in dirs), None)
print('VISOB data:', VISOB)
if VISOB is None:
    problems.append('VISOB_Augmented_Recent not found under /kaggle/input: add mubashirulhassan00/visob-augmented-recent')
found = glob.glob(os.path.dirname(os.path.dirname(CKPT)) + '/**/manifest_split.csv', recursive=True)
OLD_MANIFEST = found[0] if len(found) == 1 else None
print('training-time split:', found)
if OLD_MANIFEST is None:
    problems.append('expected exactly one manifest_split.csv in the checkpoint dataset (the training-time split)')
if WEIGHTS is not None:
    if not os.path.exists(WEIGHTS):
        problems.append(f'WEIGHTS = {WEIGHTS} not found')
else:
    code, weights_url = py('from src.xai.seg_masks import WEIGHTS_URL; print(WEIGHTS_URL)')
    r = subprocess.run(['curl', '-sSI', '--max-time', '30', '-o', '/dev/null', '-w', '%{http_code}', weights_url],
                       capture_output=True, text=True)
    print('eye-mask weights:', weights_url, '| HTTP', r.stdout.strip())
    if code or r.returncode or r.stdout.strip()[:1] not in ('2', '3'):
        problems.append('cannot reach the eye-mask weights: turn Internet on, or attach combine.pth and set WEIGHTS')
free = shutil.disk_usage(W).free / 2**30
print(f'free space in {W}: {free:.1f} GB')
if free < 3:
    problems.append(f'only {free:.1f} GB free in {W}')
print('\n'.join(['PROBLEMS:'] + problems) if problems else f'pre-flight ok ({hours():.2f} h into the session)')
assert not problems, 'fix the problems above, then run this cell again'

In [ ]:
# 7. Manifest, the pinned random split, the open-set list; the test split must equal the training-time one
t0 = time.time()
FG['01_manifest'] = sh(CMD['01_manifest'])    # about 15 min
FG['02_splits'] = sh(CMD['02_splits'])        # symlink tree /tmp/ocular_recognition and manifest_split.csv
FG['02_open_set'] = sh(CMD['02_open_set'])    # splits/open_set_test.csv: the subjects outside the 300 classes (15)
print(f'data steps: {(time.time() - t0) / 60:.1f} min')

raw = pd.read_csv(f'{W}/manifest_raw.csv')
print('manifest rows:', len(raw))
assert len(raw) == 134258, 'the VISOB copy is not the version the model was trained on (134,258 images): stop'
new = pd.read_csv(f'{W}/manifest_split.csv')
old = pd.read_csv(OLD_MANIFEST)
merged = new[['relpath', 'split']].merge(old[['relpath', 'split']], on='relpath', how='outer', indicator=True)
print(merged['_merge'].value_counts().to_string())
test_new, test_old = set(new.relpath[new.split == 'test']), set(old.relpath[old.split == 'test'])
pinned = set(pd.read_csv(f'{REPO}/splits/visob_test.csv').relpath)
print('test split identical to the training-time test split:', test_new == test_old, len(test_new), len(test_old))
assert test_new == test_old == pinned and len(pinned) == 13868, 'the test split differs from training: stop'
print(f'open-set list: {len(pd.read_csv(OPEN_SET))} images of subjects outside the 300 classes')

## 8-9. Phase A: clean evaluation, eye masks, verification smoke run

GPU 0 runs 04 (about 31 min), which writes the list of correctly classified test images that 05 and 14 sample from. GPU 1 makes the eye masks (17: a 40-image smoke run, then all 13,868 test images, about 10-20 min; the weights are downloaded and sha256-checked) and runs the verification smoke run (15, 10 subjects per protocol). Look at the two mask overlays in cell 9 before trusting the region columns: the masks are zero-shot and are only validated once hand labels exist (`scripts/18_validate_masks.py`, `docs/mask_labelling.md`).

In [ ]:
# 8. Phase A: 04 on GPU 0; 17 smoke, 17 on all test images, then the 15 smoke run on GPU 1
A = wait([launch('GPU 0', ['04_eval'], gpu=0), launch('GPU 1', ['17_smoke', '17_masks', '15_smoke'], gpu=1)],
         every=120)

In [ ]:
# 9. Check Phase A: 13,585 correct images, the eye-mask file (and its overlays), the verification smoke run;
#    then the labelling pack
from IPython.display import Image, display
assert A['04_eval'] == 0, 'the clean evaluation failed: see its log above'
correct = pd.read_csv(CORRECT)
failed = report(run_checks((checks_04,), (checks_masks, correct)))
if A['17_masks'] == 0:
    ms = load(f'{W}/eye_masks/mask_summary_test.json')
    print(f"eye masks: {ms['n_images']} images in {ms['seconds_segmenting'] / 60:.1f} min | median area shares:",
          {c: round(q['p50'], 3) for c, q in ms['area_quantiles'].items()})
    for png in ('mask_qc_test.png', 'mask_qc_low_iris_test.png'):
        if os.path.exists(f'{W}/eye_masks/{png}'):
            display(Image(filename=f'{W}/eye_masks/{png}'))
if EXPORT_LABEL_PACK and A['17_masks'] == 0:
    # the labelling pack right away (seed 0: the same 60 images in every run), so labelling can start now
    FG['18_export'] = sh(CMD['18_export'], check=False)   # label_pack.zip: VISOB images, keep it private
if A['15_smoke'] != 0:
    print('WARNING: the 15 smoke run failed, so 15 will not run in Phase C (see logs/15_smoke.log)')
assert not failed, ("fix the failed checks above before the smoke runs (if only the eye masks failed, "
                    "MASK_SOURCE = 'heuristic' runs 05 without them)")

## 10-11. Phase B: smoke runs, checks and time estimates

05 and 14 run on the same 8 images (seed 1, as the 25 and 27 Oct trials did), and the full runs leave them out. The 05 smoke rows also go through 12 (quick settings) so that the analysis is tested before the long runs. Cell 11 checks the smoke outputs, prints the estimates for Phase C and refuses to start the full runs if they would end after `HARD_STOP_H`. The estimates are upper bounds: the smoke runs include the warm-up and use smaller batches (on 25 Oct the 8-image trial predicted 4.1 h for a 05 run that took 2.2 h).

In [ ]:
# 10. Phase B: 05 smoke on GPU 0, 14 smoke on GPU 1, 16 smoke on the CPU; then 12 on the 05 smoke rows
B = wait([launch('GPU 0', ['05_smoke'], gpu=0), launch('GPU 1', ['14_smoke'], gpu=1), launch('CPU', ['16_smoke'])])
FG['12_smoke'] = sh(CMD['12_smoke'], check=False) if B['05_smoke'] == 0 else None
if FG['12_smoke'] not in (0, None):
    print('WARNING: 12 failed on the smoke rows; the full rows can still be analysed on a laptop')

In [ ]:
# 11. Smoke checks, time estimates for Phase C and the 12-hour check
images_smoke = pd.read_csv(f'{EXP}/images_{SMOKE}.csv') if os.path.exists(f'{EXP}/images_{SMOKE}.csv') else None
smoke_failed = report(run_checks((checks_05, SMOKE, 8), (checks_14, SMOKE, images_smoke), (checks_15, SMOKE, False),
                                 (checks_16, f'{W}/data_check_smoke', 20)))
assert B['05_smoke'] == 0 and B['14_smoke'] == 0, '05 or 14 failed in the smoke run: fix it before the full runs'
assert not [c for c in smoke_failed if c[0].startswith(('05', '14'))], 'fix the 05 / 14 problems above first'

s05, s14 = load(f'{EXP}/summary_{SMOKE}.json'), load(f'{ADAPTIVE}/summary_adaptive_{SMOKE}.json')
rates = lambda table: {k: None if v['rate'] is None else round(v['rate'], 2) for k, v in table.items()}
print('\n05 smoke: attack success', rates(s05['attack_success_rate']))
print('05 smoke: monocle placement', s05['monocle_placement_counts'], '| CW perturbation',
      s05['perturbation_size'].get('cw'))
print('14 smoke: success after the defence', rates(s14['success_after_defence']))

EST = {'05_full': (s05['minutes'] / s05['n_images'] * 300 * PER_CLASS_05 + 3) / 60,
       '14_full': (s14['minutes'] / s14['n_images'] * 300 + 3) / 60}
if A['15_smoke'] == 0:
    v = load(f'{VERIF}/verification_summary_{SMOKE}.json')
    n15 = 13868 + len(pd.read_csv(OPEN_SET))
    EST['15_full'] = (v['timing']['embedding_s'] / v['n_images_embedded'] * n15 + v['timing']['load_model_s'] + 120) / 3600
if B['16_smoke'] == 0:
    c = load(f'{W}/data_check_smoke/summary.json')
    syn, mins = c.get('synthetic', {}), c.get('minutes', {})
    scale = syn.get('buckets_with_frames_ge_1010_total', 0) / max(syn.get('buckets_selected', 0), 1)
    EST['16_full'] = (mins.get('counts', 0) + mins.get('overlap', 0) + mins.get('synthetic', 0) * scale
                      + mins.get('areas', 0) / max(c.get('areas', {}).get('images', 0), 1) * len(correct)) / 60
phase_c = max(EST['05_full'], EST['14_full'] + EST.get('15_full', 0))
finish = hours() + phase_c + 0.4          # + about 25 min for the analysis, results.zip and the checklist
for name, h in EST.items():
    print(f'{name}: about {h:.1f} h (upper bound)')
print(f'Phase C on the GPUs: about {phase_c:.1f} h; the session would end at about {finish:.1f} h of the 12 '
      f'(16 runs on the CPU at the same time and is stopped at {HARD_STOP_H} h if it is still running)')
if RUN_FULL:
    assert finish < HARD_STOP_H, ('the full runs would not fit in the session: lower PER_CLASS_05, drop cw from '
                                  'ATTACKS_05 or lower ADAPTIVE_STEPS, then run again')

## 12-13. Phase C and D: the full runs and the analysis (`RUN_FULL = True`)

GPU 0 runs 05 on 600 images (seed 2, the smoke images left out: the same 600 images as the 25 and 27 Oct runs when the correct-image list is unchanged), with eye-mask regions and every map saved. GPU 1 runs 14 on 300 of those images (one per subject, paired with 05) and then 15 (P1, inflated, reproduction only; P3, open-set and cross-session). The CPU runs the data check (16) at low priority, including the per-image heuristic areas of every correct test image. Then 12 analyses the new rows.

In [ ]:
# 12. Phase C (RUN_FULL): 05 full on GPU 0; 14 full, then 15 P1 + P3, on GPU 1; 16 on the CPU
C = {}
if RUN_FULL:
    gpu_jobs = [launch('GPU 0', ['05_full'], gpu=0), launch('GPU 1', ['14_full', '15_full'], gpu=1)]
    cpu_job = launch('CPU', ['16_full'])
    C = wait(gpu_jobs, every=300, also=[cpu_job])
else:
    print('RUN_FULL is False: full runs skipped. Set RUN_FULL = True (cell 2) and use Save Version > Save & Run All.')

In [ ]:
# 13. Phase D (RUN_FULL): the v2 analysis of the new rows, then the data check's end
if RUN_FULL:
    if C.get('05_full') == 0 and hours() < HARD_STOP_H - 0.3:
        FG['12_full'] = sh(CMD['12_full'], check=False)    # about 5-10 min
    else:
        print('12 skipped: 05 did not finish, or too little time is left (run it on a laptop from the rows)')
    D = wait([cpu_job], every=300)

## 14-15. results.zip and the final checklist

`results.zip` is written before the checklist, so a failed check never loses the outputs. It holds every output folder and the logs, except files above `ZIP_MAX_MB` (the 15 embeddings) and the labelling pack (download those separately if needed). The checklist asserts the expected files and the key sanity checks: budgets exact, no missing eye masks, counts, commits, pairing of 14 with 05.

In [ ]:
# 14. results.zip (download it from the Output panel; never commit or upload it)
import zipfile
info = {'branch': BRANCH, 'commit': COMMIT, 'run_full': RUN_FULL, 'tag': TAG, 'smoke_tag': SMOKE,
        'settings': {'levels': LEVELS, 'per_class_05': PER_CLASS_05, 'attacks_05': ATTACKS_05, 'cw_kappa': CW_KAPPA,
                     'jpeg_quality': JPEG_QUALITY, 'bits': BITS, 'mask_source': MASK_SOURCE,
                     'adaptive_steps': ADAPTIVE_STEPS, 'batch': BATCH, 'data_check_workers': DATA_CHECK_WORKERS},
        'commands': CMD, 'steps': {**{n: status(n) for n in CMD if n not in FG}, **FG},
        'estimates_h': globals().get('EST', {}), 'session_hours': round(hours(), 2)}
with open(f'{W}/run_info.json', 'w') as f:
    json.dump(info, f, indent=2)
folders = ['eval_clean', 'eye_masks', 'experiment', 'adaptive', 'verification', 'data_check', 'data_check_smoke',
           f'analysis_{SMOKE}', f'analysis_{TAG}', 'splits', 'logs']
left_out = []
with zipfile.ZipFile(f'{W}/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(f'{W}/run_info.json', 'run_info.json')
    for folder in folders:
        for root, _, files in os.walk(f'{W}/{folder}'):
            for name in sorted(files):
                path = os.path.join(root, name)
                if os.path.getsize(path) > ZIP_MAX_MB * 2**20:
                    left_out.append(os.path.relpath(path, W))
                else:
                    z.write(path, os.path.relpath(path, W))
print(f"results.zip: {os.path.getsize(f'{W}/results.zip') / 2**20:.0f} MB")
if left_out:
    print(f'left out (larger than {ZIP_MAX_MB} MB, still in the Output panel):', left_out)

In [ ]:
# 15. Final checklist: expected files and sanity checks; the run fails here if a hard check fails
correct = pd.read_csv(CORRECT) if os.path.exists(CORRECT) else None
n_correct = None if correct is None else len(correct)
tag = TAG if RUN_FULL else SMOKE
images_05 = pd.read_csv(f'{EXP}/images_{tag}.csv') if os.path.exists(f'{EXP}/images_{tag}.csv') else None
if RUN_FULL:
    checks = run_checks((checks_04,), (checks_masks, correct), (checks_05, TAG, 300 * PER_CLASS_05),
                        (checks_14, TAG, images_05), (checks_15, TAG, True), (checks_16, CHECK, n_correct),
                        (checks_12, f'{W}/analysis_{TAG}', True))
    expected = ['01_manifest', '02_splits', '02_open_set', '04_eval', '17_smoke', '17_masks', '15_smoke', '05_smoke',
                '14_smoke', '16_smoke', '05_full', '14_full', '15_full', '16_full', '12_full']
else:
    checks = run_checks((checks_04,), (checks_masks, correct), (checks_05, SMOKE, 8), (checks_14, SMOKE, images_05),
                        (checks_15, SMOKE, False), (checks_16, f'{W}/data_check_smoke', 20),
                        (checks_12, f'{W}/analysis_{SMOKE}', False))
    expected = ['01_manifest', '02_splits', '02_open_set', '04_eval', '17_smoke', '17_masks', '15_smoke', '05_smoke',
                '14_smoke', '16_smoke']
if '18_export' in FG:
    expected.append('18_export')
steps = {n: FG.get(n, status(n)) for n in expected}
bad = {n: s for n, s in steps.items() if s != 0}
checks.append(('every step of the session succeeded', not bad, ', '.join(f'{n}: {s}' for n, s in bad.items()), True))
checks.append(('results.zip written', os.path.exists(f'{W}/results.zip'), '', True))
failed = report(checks)
warned = [c for c in checks if not c[1] and not c[3]]
record = {'run_full': RUN_FULL, 'commit': COMMIT, 'session_hours': round(hours(), 2), 'steps': steps,
          'checks': [{'check': n, 'passed': bool(ok), 'detail': d, 'hard': h} for n, ok, d, h in checks]}
name = f'checklist_{tag}.json'
with open(f'{W}/{name}', 'w') as f:
    json.dump(record, f, indent=2)
with zipfile.ZipFile(f'{W}/results.zip', 'a') as z:
    z.write(f'{W}/{name}', name)
print(f'\n{len(checks) - len(failed) - len(warned)} passed, {len(warned)} warnings, {len(failed)} failed | '
      f'session {hours():.1f} h')
assert not failed, 'some checks failed (listed above); the outputs and results.zip are saved'

## After the run

1. Download `results.zip` (and, if needed, `verification/embeddings_v2.npz`) and unzip it into a new folder under `D:\TempCLG\results` on the laptop. Never commit it or put it on the web; back it up in the team's private storage.
2. Read `data_check/report.md`. If frames numbered 1010 or higher turn out to be synthetic copies, the team decides whether to leave them out; the rows carry `relpath`, so they can be filtered without a new run.
3. The masks stay unvalidated until hand labels exist: look at `eye_masks/mask_qc_test.png`, then download `label_pack.zip` from the Output panel (written right after Phase A, about 1 h into the session; it is not in `results.zip`) and label it as `docs/mask_labelling.md` explains.
4. On the laptop, redo the 27 Oct analysis with exact per-image areas: `python scripts/12_analyse.py --experiment-rows D:/TempCLG/results/rows_full.csv --areas <unzipped>/data_check/areas_correct_test_full.csv --out-dir <new folder>`.
5. If the checklist says so, 05 used the same 600 images as the 25 and 27 Oct runs, and old and new rows can be compared image by image.